# 02. EDA - Khám phá, kiểm tra và làm sạch

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / 'src/ml_scratch').is_dir():
    if ROOT == ROOT.parent: raise FileNotFoundError('Mở notebook trong dự án')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import display, Image


## Vai trò EDA
mục đích: sau khi đọc raw, dữ liệu có thiếu, rỗng, trùng, lệch lớp, hay độ dài bất thường không?

EDA làm đúng mục đích đó và dừng ở hiểu biết. Feature engineering ở bước 03 mới biến `clean_text` thành vector TF-IDF. Hai việc khác nhau: bước này không chọn K hay alpha, không fit model, không nhìn test để quyết định cách làm sạch.

`word_count` và `char_count` chỉ để nhìn phân bố. Model hiện tại không nhận hai cột này. Stop words cũng chưa bỏ ở đây; vectorizer ở bước 03 mới lo.

In [ ]:
from ml_scratch.data import prepare
from ml_scratch.evaluation import eda
raw, data = prepare()
eda(raw, data)
display(data.groupby("label")[["word_count", "char_count"]].describe())
print("Raw rows", len(raw), "clean rows", len(data), "removed", len(raw) - len(data))
sample = data.groupby("label", as_index=False).head(1)[["label", "word_count", "char_count", "clean_text"]].copy()
sample["clean_text"] = sample.clean_text.str.slice(0, 180)
display(sample)

## Thống kê và biểu đồ theo loại biến
`label` là phân loại: count, tỷ lệ, bar chart `data_labels.png` (số ghi trên cột). Không vẽ histogram cho nhãn.

`word_count` và `char_count` là số: mean, std, min, Q1, median, Q3, max trong `outputs/reports/numeric_summary.csv`. `data_lengths.png` có histogram và boxplot của cả hai, tách theo ham/spam. Mean lớn hơn median vì vài thư rất dài. Không cắt outlier: email dài vẫn là email thật, và histogram đủ dải để thấy đuôi đó.

`text` là chuỗi: đếm rỗng và trùng sau `clean_text`, xem một dòng ví dụ mỗi lớp. Không có trung bình của chuỗi.

`source_row` giữ chỉ mục raw để đối chiếu lỗi sau này. `label_num` không có trong file sạch vì đã dùng `label`.

Làm sạch trong `prepare()`: lowercase, bỏ HTML, thay ký tự ngoài chữ/số/khoảng trắng, gộp khoảng trắng, bỏ nội dung rỗng, bỏ trùng đúng từng ký tự. Trùng khác nhãn thì hàm báo lỗi.

In [ ]:
display(Image(filename=str(ROOT/"outputs/figures/data_labels.png")))
display(Image(filename=str(ROOT/"outputs/figures/data_lengths.png")))

## Đọc số và handoff
Raw 5.171 email, không thiếu. Sau làm sạch còn 4.888: 3.427 ham (70,11%) và 1.461 spam (29,89%). 283 bản là trùng chính xác và bị loại; không có thư rỗng sau clean. Tỷ lệ ham gần như không đổi, nên bước 03 phải stratify.

Độ dài lệch phải. Ham median 87 từ nhưng mean khoảng 159 (max 3.568). Spam median 91 từ, mean khoảng 190 (max 2.602). Ký tự cũng vậy: ham median 449, spam median 511, cả hai có thư trên 20.000 ký tự. Histogram dồn về bên trái vì đuôi dài; boxplot và quartile cho thấy hai lớp chồng nhau. Độ dài vì vậy không vào model.

Output: `data/processed/clean_emails.csv` với `source_row`, `text`, `label`, `clean_text`, `word_count`, `char_count`. Bước 03 chỉ đọc file này. Thêm `outputs/reports/label_distribution.csv`, `numeric_summary.csv`, `outputs/figures/data_labels.png`, `data_lengths.png`.

Notebook 01 là biên bản kiểm kê. Notebook này gọi `prepare()`, và hàm đó tự chạy lại collection, nên làm sạch vẫn đúng nếu chạy 02 riêng.